# Restore Grid World


In [16]:
import torch 
import json 
from gridworld import GridWorld

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device) # cuda

with open("gridworld.json", "r") as f:
    json_object = json.load(f)
    env = GridWorld(**json_object, device=device)

cuda


In [17]:
print(env.valid_states)
print(env.grid_size)

tensor([[48, 49],
        [47, 49],
        [47, 48],
        ...,
        [43, 49],
        [44, 49],
        [45, 49]], device='cuda:0')
50


## 1.0 Visualize Grid World

In [18]:
from IPython.display import HTML, display 

valid_states_all = torch.cat((env.valid_states, torch.tensor([env.grid_size-1,env.grid_size-1]).to(device).unsqueeze(dim=0)))

valid_states_all = valid_states_all[:,0]*env.grid_size + valid_states_all[:,1]

# print(valid_states_all)
# print(valid_states_all[:,0])
# print(valid_states_all[:,1])

html_text = "<table>" 
for row in range(env.grid_size):
    html_text += "<tr>"
    for col in range(env.grid_size):
        if row*env.grid_size + col in valid_states_all:
            html_text += "<td bgcolor=\"pearl\">"
        else:
            html_text += "<td>"
        html_text += str(env.reward_map[row*env.grid_size+col].tolist())
        html_text += "</td>"
    html_text += "</tr>"
html_text += "</table>"

display(HTML(html_text))

0,-1,0,0,0,-1,0,0,0,0,0,0,-1,0,-1,-1,-1,-1,0,-1,0,-1,0,0,0,-1,0,0,0,-1,-1,0,-1,0,0,0,0,-1,-1,0,0,0,0,0,0,0,0,0,0,0
-1,0,0,0,0,-1,0,-1,0,0,0,0,0,0,0,0,0,0,0,0,0,-1,-1,-1,0,-1,0,0,0,0,0,-1,0,-1,0,-1,0,-1,0,-1,-1,0,0,0,-1,-1,0,0,-1,-1
0,0,-1,-1,0,-1,0,-1,0,-1,0,0,0,0,0,0,0,-1,0,0,-1,0,0,-1,0,0,0,0,0,0,0,-1,0,0,-1,0,0,-1,-1,0,0,-1,0,0,0,0,-1,0,0,0
0,0,-1,0,-1,-1,0,0,-1,0,-1,0,0,0,0,-1,0,-1,0,-1,0,-1,0,0,-1,-1,0,0,-1,0,0,0,0,0,0,0,0,0,0,0,0,-1,0,0,0,0,0,0,0,-1
-1,-1,-1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-1,-1,0,-1,0,0,0,0,0,0,0,0,0,0,0,0,-1,0,-1,0,0,0,-1,0,0,-1,0,-1,-1,0,0
0,-1,-1,0,0,-1,0,-1,0,0,-1,0,0,-1,0,0,-1,-1,0,0,-1,0,0,0,-1,0,0,0,0,-1,-1,0,0,-1,0,-1,-1,-1,-1,0,0,-1,0,-1,0,0,0,0,0,-1
-1,-1,0,0,0,0,0,0,0,0,0,0,-1,0,-1,0,0,0,0,0,-1,-1,0,0,-1,-1,0,0,-1,-1,0,-1,0,0,0,0,0,0,0,-1,-1,0,0,0,-1,0,-1,0,-1,-1
0,0,0,0,-1,0,0,0,-1,0,0,0,0,-1,0,0,-1,-1,0,0,-1,0,0,-1,0,0,-1,0,0,0,-1,-1,0,0,-1,0,-1,0,-1,0,0,-1,0,-1,0,0,0,0,0,0
0,0,0,0,0,0,0,0,0,0,-1,-1,0,-1,0,0,-1,0,0,-1,-1,0,0,0,0,0,0,-1,0,0,0,-1,-1,-1,-1,-1,-1,0,-1,-1,0,-1,0,0,0,-1,0,0,0,0
0,0,0,-1,0,-1,0,0,-1,-1,0,0,-1,0,0,0,0,0,0,0,0,0,-1,-1,-1,0,-1,0,0,0,0,0,0,0,0,0,0,-1,0,0,0,0,0,0,-1,-1,0,-1,-1,-1
0,0,0,0,0,-1,0,0,0,0,-1,0,0,-1,0,0,0,-1,0,0,0,-1,0,0,-1,0,0,0,-1,-1,0,-1,-1,0,0,0,0,0,-1,-1,-1,-1,-1,-1,0,0,-1,-1,-1,0


# Define Policy

Build a policy $\pi_{\theta}$

In [19]:
import torch.nn as nn 
from torch.nn import functional as F 

STATE_SIZE = env.grid_size*env.grid_size # total 2500 square
ACTION_SIZE = env.action_size  # 4 options to choose 

#
# Define model
# 

class PolicyNet(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.hidden = nn.Linear(STATE_SIZE, hidden_dim) # 2500 x 64
        self.classify = nn.Linear(hidden_dim, ACTION_SIZE) # 64 x 4 
    
    def forward(self, s): 
        outs = self.hidden(s)
        outs = F.relu(outs)
        logits = self.classify(outs)
        return logits 

#
# Generate model
# 

policy_func = PolicyNet().to(device)

In [20]:
print(policy_func)

PolicyNet(
  (hidden): Linear(in_features=2500, out_features=64, bias=True)
  (classify): Linear(in_features=64, out_features=4, bias=True)
)


# Run before train

In [21]:
def pick_sample_and_logits(policy, s):
    """
    Stochastically pick up action and logits with policy model.

    Parameters
    ----------
    policy: torch.nn.Module
            Policy network to use
    s: torch.tensor((..., STATE_SIZE), dtype=int)
            The feature (one-hot) of state
            The above "..." can have arbitrary share with 0 or 1 dimension.
    
    Returns
    -------
    action: torch.tensor((...), dtype=int)
            The picked-up actions
            If input shape is (*, STATE_SIZE), this shape becomes (*).
    logits: torch.tensor((..., ACTION_SIZE), dtype=float)
            Logits of categorical distribution (used to optimize model).
            If input shape is (*, STATE_SIZE), this shape becomes (*, ACTION_SIZE).
    """
    # get logits from state
    # --> size: (*, ACTION_SIZE)
    logits = policy(s.float())

    # from logits to probabilities
    # --> size: (* ACTION_SIZE)
    probs = F.softmax(logits, dim=-1)

    # pick up action's sample
    # --> size:(*, 1)
    a = torch.multinomial(probs, num_samples=1)
    
    # --> size: (*)
    a = a.squeeze()

    return a, logits

In [22]:
s = env.reset(1)
print(f"get Raw state: {s[0]}")

s_onehot = F.one_hot(s, num_classes=STATE_SIZE)
print(f"shape of one hot: {s_onehot.shape}")
print(f"one hot: {s_onehot}")

device = "cuda"
policy = PolicyNet().to(device)

logits = policy(s_onehot.float())
probs = F.softmax(logits, dim=-1)
print(f"logits shape: {logits.shape}")
print(f"logits are calculated: {logits}")
print(f"probss are calculated: {probs}")

action = torch.multinomial(probs, num_samples=1)
action = action.squeeze()
print(f"Action has been selected: {action}")


get Raw state: 1031
shape of one hot: torch.Size([1, 2500])
one hot: tensor([[0, 0, 0,  ..., 0, 0, 0]], device='cuda:0')
logits shape: torch.Size([1, 4])
logits are calculated: tensor([[ 0.0721,  0.1180, -0.0589,  0.0880]], device='cuda:0',
       grad_fn=<AddmmBackward0>)
probss are calculated: tensor([[0.2538, 0.2657, 0.2226, 0.2579]], device='cuda:0',
       grad_fn=<SoftmaxBackward0>)
Action has been selected: 2


In [23]:
action = torch.multinomial(probs, num_samples=1)
action = action.squeeze()
print(f"One u selected")
print(f"Action has been selected: {action}")

action_1000 = torch.multinomial(probs, num_samples=1000, replacement=True)
action = action_1000.squeeze()
print("="*10)
print(f"1000 u selected")
print(f"Action has been selected: {action}")

counts = torch.bincount(action)

print(f"Counts for each action (0 to 3): {counts}")

# 3. Optional: Convert counts to percentages to check against your probabilities
percentages = counts.float() / 1000
print(f"Empirical probabilities: {percentages}")
print(f"Original network probabilities: {probs}")

One u selected
Action has been selected: 1
1000 u selected
Action has been selected: tensor([1, 0, 0, 0, 0, 3, 3, 3, 2, 2, 0, 1, 1, 3, 1, 0, 1, 3, 1, 3, 3, 3, 2, 3,
        0, 2, 1, 3, 1, 2, 0, 0, 1, 0, 0, 1, 1, 1, 3, 0, 0, 1, 3, 3, 2, 2, 1, 0,
        0, 1, 0, 3, 2, 0, 2, 0, 3, 2, 1, 3, 2, 0, 3, 0, 0, 2, 3, 2, 3, 1, 2, 0,
        2, 1, 3, 2, 2, 3, 2, 1, 0, 2, 1, 3, 2, 1, 3, 0, 1, 0, 0, 2, 1, 2, 1, 0,
        2, 1, 0, 1, 2, 1, 0, 1, 0, 1, 3, 1, 3, 3, 2, 3, 1, 1, 2, 3, 0, 0, 3, 0,
        0, 0, 3, 3, 0, 1, 2, 0, 2, 1, 3, 3, 2, 3, 3, 2, 2, 0, 0, 2, 0, 1, 1, 1,
        1, 1, 3, 3, 3, 1, 1, 3, 0, 2, 2, 1, 3, 2, 1, 1, 3, 1, 1, 1, 3, 1, 0, 2,
        3, 1, 3, 3, 0, 2, 0, 3, 0, 2, 3, 0, 2, 2, 1, 1, 3, 3, 3, 2, 0, 2, 3, 1,
        0, 0, 3, 2, 0, 3, 1, 3, 0, 1, 2, 1, 1, 1, 1, 2, 3, 0, 0, 1, 2, 2, 1, 3,
        3, 3, 2, 1, 0, 1, 2, 3, 0, 1, 0, 0, 0, 3, 1, 0, 0, 1, 0, 2, 1, 0, 0, 0,
        0, 3, 1, 3, 3, 2, 3, 0, 2, 3, 3, 1, 1, 0, 2, 1, 2, 2, 0, 3, 1, 3, 2, 3,
        2, 0, 1, 2, 3, 2, 2, 2, 3, 

In [24]:
def evaluate(policy, batch_size):
    total_reward = torch.tensor(0.0).to(device) # init reward = 0.0
    s = env.reset(batch_size)
    # print(f"state: {s.shape}")
    i = 0
    while True:
        i+=1
        s_onehot = F.one_hot(s, num_classes=STATE_SIZE)
        
        with torch.no_grad():
            a, _ = pick_sample_and_logits(policy, s_onehot)
        
        s, r, term, trunc = env.step(a, s)
        # --> s = pi(a,s) next step calculated from recent step s and next action
        # --> r: reward while hitting next step
        # --> term: check if the agent terminates or not
        # --> trunc: check if the time run out or not
        
        total_reward += torch.sum(r)
        # reward sum
        
        done = torch.logical_or(term, trunc)
        # check if the agent done or not
        # term trunc done
        # False False False
        # True False True
        # False True True
        # True True True
        work_indices = (done==False).nonzero().squeeze(dim=-1)
        if not (len(work_indices) > 0):
            break
        s = s[work_indices]
    # print(f"iteration: {i}")
    return total_reward.item() / batch_size 

avg_reward = evaluate(policy_func, 300)
# print(f"Estimate Reward: {avg_reward}")

# Train Policy

## 0.0 sample code

In [28]:
import pickle 

# Dataset
dest_dir = "./expert_data"
checkpoint_files = ["ckpt0.pkl"]

# create optimizer 
opt = torch.optim.AdamW(policy_func.parameters(), lr=0.001)

for ckpt in checkpoint_files:
    # Load expert data from pickle
    with open(f"{dest_dir}/{ckpt}", "rb") as f:
        all_data = pickle.load(f)

    all_states = all_data["states"]
    all_actions = all_data["actions"]
    timestep_lens = all_data["timestep_lens"]

    # Loop all episodes in demonstration
    current_timestep = 0 
    for i, timestep_len in enumerate(timestep_lens): 
        # pick up states and actions in a single episode 
        states = all_states[current_timestep:current_timestep+timestep_len]
        actions = all_actions[current_timestep:current_timestep+timestep_len]
        # print(f"States:\n{states}")
        # print(f"Actions:\n{actions}")
        # # collect loss and optimize (train)
        opt.zero_grad()
        loss = []

        for s, a in zip(states, actions):
            s_onehot = F.one_hot(torch.tensor(s).to(device), num_classes=STATE_SIZE)
            _, logits = pick_sample_and_logits(policy_func, s_onehot)
            l = F.cross_entropy(logits, torch.tensor(a).to(device), reduction='none')
            loss.append(l)
        total_loss = torch.stack(loss, dim=0)
        total_loss.sum().backward()
        opt.step()

        # log
        print("Processed {:5d} episodes in checkpoint {}...".format(i + 1, ckpt), end="\r")

        # run evaluation in each 1000 episodes 
        if i % 1000 == 999: 
            avg = evaluate(policy_func, 200)
            print(f"\nEvaluation result (Average reward): {avg}")

        # proceed to next episode
        current_timestep += timestep_len

        torch.save({
    'model_state_dict': policy_func.state_dict(),
    'optimizer_state_dict': opt.state_dict(),
}, "model_train.pt")

Processed  1000 episodes in checkpoint ckpt0.pkl...
Evaluation result (Average reward): 8.505
Processed  2000 episodes in checkpoint ckpt0.pkl...
Evaluation result (Average reward): 8.605
Processed  3000 episodes in checkpoint ckpt0.pkl...
Evaluation result (Average reward): 8.395
Processed  4000 episodes in checkpoint ckpt0.pkl...
Evaluation result (Average reward): 8.53
Processed  5000 episodes in checkpoint ckpt0.pkl...
Evaluation result (Average reward): 8.155
Processed  6000 episodes in checkpoint ckpt0.pkl...
Evaluation result (Average reward): 8.95
Processed  7000 episodes in checkpoint ckpt0.pkl...
Evaluation result (Average reward): 8.565
Processed  8000 episodes in checkpoint ckpt0.pkl...
Evaluation result (Average reward): 8.49
Processed  9000 episodes in checkpoint ckpt0.pkl...
Evaluation result (Average reward): 8.715
Processed 10000 episodes in checkpoint ckpt0.pkl...
Evaluation result (Average reward): 8.745


## 1.0 Explore dataset